In [1]:
# !pip install pinecone sentence-transformers
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

In [2]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
pinecone_api_key = os.getenv("PINECONE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

In [4]:
#Connect to the pinecone
pinecone = Pinecone(api_key=pinecone_api_key)

In [5]:
pinecone

Pinecone(api_key='...KRca', host='https://api.pinecone.io')

In [6]:
INDEX_NAME = "ragtestv1"

pinecone.create_index(
    name=INDEX_NAME,
    dimension=384,  #Depends on your embedding Model
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

IndexModel(name='ragtestv1', metric='cosine', host='https://ragtestv1-503dlyx.svc.aped-4627-b74a.pinecone.io', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), vector_type='dense', dimension=384, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [7]:
# Created API key and Created the index
# some text and convert to embeddings then insert into the vector db index

In [8]:
# ---------- upsert: simple documents ----------
# Step 3: Chunking and organizing documents - Here I'm giving you already organized docs.
docs = [
    {"id": "id1", "text": "Pandas is a Python library for data analysis."},
    {"id": "id2", "text": "Pinecone is a vector database for semantic search."},
    {"id": "id3", "text": "Spark enables distributed data processing."},
]

docs

[{'id': 'id1', 'text': 'Pandas is a Python library for data analysis.'},
 {'id': 'id2', 'text': 'Pinecone is a vector database for semantic search.'},
 {'id': 'id3', 'text': 'Spark enables distributed data processing.'}]

In [9]:
# embed in a small batch (better for throughput than per-doc calls)
# Step 4 - Creating Embeddings
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")  # 384-dim
embeddings = model.encode([d["text"] for d in docs])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
len(embeddings[1])

384

In [11]:
embeddings[0]

array([-4.88402337e-01, -7.72713363e-01, -1.79367423e-01,  2.75081277e-01,
        1.05562091e-01, -2.61944443e-01,  5.11257887e-01, -1.22027636e-01,
        1.51373312e-01,  4.51361090e-01, -1.83275506e-01, -3.89374226e-01,
        2.13705018e-01,  3.72340590e-01, -4.15670484e-01, -2.12128926e-02,
        3.46354634e-01, -5.76614030e-02, -2.83594728e-02, -5.01845777e-01,
       -4.71999884e-01, -5.06300665e-02, -2.25893274e-01,  3.88020515e-01,
       -2.10162655e-01, -5.84237397e-01, -2.57924408e-01, -4.91476357e-02,
        6.05109744e-02,  4.01333012e-02,  1.42069474e-01, -2.38506794e-01,
       -6.56093955e-02,  3.72613788e-01, -4.34653550e-01,  2.55043656e-01,
       -5.17540090e-02, -3.30621362e-01, -1.25788167e-01,  1.37097389e-01,
       -3.84487152e-01,  3.62321138e-01,  6.59914553e-01,  2.60308325e-01,
        1.28738403e-01, -6.78920299e-02, -2.76502877e-01, -2.70921350e-01,
       -3.35516334e-01,  1.44570291e-01, -2.01469004e-01,  1.21743359e-01,
       -4.19496387e-01, -

In [12]:
# Step 6 - Organize vectors with ids/metadata
vectors = [
    {
        "id": d["id"],
        "values": emb.tolist(),
        "metadata": {"text": d["text"]},
    }
    for d, emb in zip(docs, embeddings)
]

In [13]:
vectors

[{'id': 'id1',
  'values': [-0.4884023368358612,
   -0.7727133631706238,
   -0.17936742305755615,
   0.2750812768936157,
   0.10556209087371826,
   -0.261944442987442,
   0.5112578868865967,
   -0.12202763557434082,
   0.15137331187725067,
   0.4513610899448395,
   -0.183275505900383,
   -0.3893742263317108,
   0.21370501816272736,
   0.372340589761734,
   -0.4156704843044281,
   -0.02121289260685444,
   0.3463546335697174,
   -0.057661402970552444,
   -0.02835947275161743,
   -0.5018457770347595,
   -0.4719998836517334,
   -0.05063006654381752,
   -0.2258932739496231,
   0.38802051544189453,
   -0.21016265451908112,
   -0.5842373967170715,
   -0.2579244077205658,
   -0.04914763569831848,
   0.06051097437739372,
   0.04013330116868019,
   0.142069473862648,
   -0.23850679397583008,
   -0.0656093955039978,
   0.372613787651062,
   -0.43465355038642883,
   0.2550436556339264,
   -0.05175400897860527,
   -0.3306213617324829,
   -0.12578816711902618,
   0.13709738850593567,
   -0.384487152

In [14]:
INDEX_NAME = "ragtestv1"
index = pinecone.Index(INDEX_NAME)
index

Index(host='https://ragtestv1-503dlyx.svc.aped-4627-b74a.pinecone.io')

In [15]:
# Used for inserting the into the vector db
index.upsert(vectors)

TypeError: Index.upsert() takes 1 positional argument but 2 were given

In [16]:
# User query
# Input query
query = "what tool we should use for data analysis"
qvec = model.encode([query])[0].tolist()
len(qvec)

384

In [17]:
# get the closest value from vector DB
res = index.query(vector=qvec, top_k=3, include_metadata=True)
res

QueryResponse(matches=[], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sat, 16 May 2026 18:41:45 GMT', 'content-type': 'application/json', 'content-length': '66', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '65', 'x-envoy-upstream-service-time': '64', 'x-pinecone-response-duration-ms': '67', 'grpc-status': '0', 'server': 'envoy'}))

In [18]:
# get the closest value from vector DB
res = index.query(vector=qvec, top_k=1, include_metadata=False)
res

QueryResponse(matches=[], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sat, 16 May 2026 18:41:53 GMT', 'content-type': 'application/json', 'content-length': '66', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '35', 'x-envoy-upstream-service-time': '35', 'x-pinecone-response-duration-ms': '36', 'grpc-status': '0', 'server': 'envoy'}))